# Weeks 10–11 on Kaggle: thresholds, new test sets, and the end-to-end test

**Before running:** right panel → **Session options** → Accelerator **GPU T4 x2**, Internet **on**. **Add-ons → Secrets**: `GH_TOKEN` (GitHub, `ghp_...`), `HF_TOKEN` (Hugging Face), and `OPENAI_API_KEY` (only needed for the real-agent runs in step 7). Tick the box next to each.

Every step saves to GitHub when it finishes, so a disconnect only loses the step in progress. If you get disconnected, run cell 1 again, then continue from the first step that didn't print `SAVED`.

## 1. Setup (about 2 minutes)

In [ ]:
import os, json, subprocess
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
from kaggle_secrets import UserSecretsClient
secrets = UserSecretsClient()
gh, hf = secrets.get_secret('GH_TOKEN'), secrets.get_secret('HF_TOKEN')
if not os.path.exists('/kaggle/working/pi-detector-thesis'):
    !git clone -q https://{gh}@github.com/williamsg-blip/pi-detector-thesis.git /kaggle/working/pi-detector-thesis
%cd /kaggle/working/pi-detector-thesis
!git pull -q
!git config --global user.name "Gary" && git config --global user.email "gwill90jr@gmail.com"
!pip install -q 'transformers>=4.44' 'accelerate>=0.30' sentencepiece scikit-learn agentdojo==0.1.35 python-dotenv
from huggingface_hub import login
login(token=hf)
try:
    open('.env', 'w').write('OPENAI_API_KEY=' + secrets.get_secret('OPENAI_API_KEY') + '\n')  # .env is never committed
    print('OpenAI key found: real-agent runs (step 7) are available')
except Exception:
    print('No OPENAI_API_KEY secret: skip step 7 (everything else works)')

def save(msg):
    r = subprocess.run(f'git add results/ && git commit -q -m "{msg}" && git push -q', shell=True)
    print('SAVED TO GITHUB:' if r.returncode == 0 else 'Nothing new to save, or push failed:', msg)

!nvidia-smi --query-gpu=name --format=csv,noheader
!ls data/sensitive_benign.jsonl data/val_obfuscated.jsonl data/test_obfuscated.jsonl

## 2. Retrain both DeBERTa models (about 40 minutes total)
Kaggle doesn't keep models between sessions, so they're retrained here with the same settings and seed as week 9. Results go to `results/week10_retrain_*` so your week 9 files stay untouched. If a model already exists in this session, it's skipped.

In [ ]:
%cd /kaggle/working/pi-detector-thesis
if not os.path.exists('models/deberta-v3-small/config.json'):
    !python scripts/train_classifier.py --out results/week10_retrain_deberta-v3-small | tail -4
    save('Week 10: retrained DeBERTa (chunked)')
if not os.path.exists('models/deberta-v3-small-nochunk/config.json'):
    !python scripts/train_classifier.py --no-chunking --out results/week10_retrain_deberta-v3-small-nochunk | tail -4
    save('Week 10: retrained DeBERTa (no chunking)')
!ls models/

## 3. Score every test set (about 5–10 minutes)
Scores the validation, test, sensitive-topic, and disguised sets, raw and normalized. Prompt Guard 2 is included for comparison; if its access isn't approved it's skipped.

In [ ]:
%cd /kaggle/working/pi-detector-thesis
!python scripts/score_sets.py hf:models/deberta-v3-small --name deberta-v3-small
!python scripts/score_sets.py hf:models/deberta-v3-small-nochunk --name deberta-v3-small-nochunk --no-chunking
!python scripts/score_sets.py hf:meta-llama/Llama-Prompt-Guard-2-86M --name prompt-guard-2 || echo 'Prompt Guard 2 skipped'
!python scripts/score_sets.py keyword > /dev/null && python scripts/score_sets.py tfidf > /dev/null
save('Week 10: scores for all detectors')

## 4. Week 10 analysis (seconds)

In [ ]:
%cd /kaggle/working/pi-detector-thesis
!python scripts/analyze_week10.py > /dev/null
save('Week 10: analysis and operating points')
from IPython.display import Markdown
Markdown(open('results/week10_analysis.md').read())

## 5. Pick the detector for week 11 (validation data only)

In [ ]:
points = json.load(open('results/week10_operating_points.json'))
deberta = {k: v for k, v in points.items() if k.startswith('deberta')}
DETECTOR = max(deberta, key=lambda k: deberta[k]['val_score'])
print('Using', DETECTOR, '|', deberta[DETECTOR]['variant'], '| threshold', round(deberta[DETECTOR]['threshold'], 5))

## 6. Free end-to-end test with the gullible fake agent (about 5–10 minutes)
No API calls. The fake agent follows any injection it can see, so this shows how many attacks the detector stops in the worst case, for both the training-style attack and the held-out one.

In [ ]:
%cd /kaggle/working/pi-detector-thesis
for attack in ['important_instructions', 'system_message']:
    out = f'results/week11_offline_{attack}_{DETECTOR}.csv'
    !python scripts/run_baseline.py --dry-run gullible --attack {attack} --detector {DETECTOR} --out {out} | tail -9
save('Week 11: offline end-to-end with ' + DETECTOR)

## 7. Real agent runs (needs `OPENAI_API_KEY`, costs API credits)
Ordered by importance. Each is the **banking** suite (the one the detector never trained on), 160 agent runs, about 15–30 minutes. Check your OpenAI usage after the first one.

In [ ]:
%cd /kaggle/working/pi-detector-thesis
# A. Detector vs. the same attack as your week 6 baseline (compare with 47.2% attack success on banking)
!python scripts/run_baseline.py --detector {DETECTOR} --suites banking --out results/defended_{DETECTOR}_important_instructions_banking.csv | tail -10
save('Week 11: real run A (detector, important_instructions, banking)')

In [ ]:
%cd /kaggle/working/pi-detector-thesis
# B. No defense vs. the HELD-OUT attack style (a new baseline for run C)
!python scripts/run_baseline.py --attack system_message --suites banking --out results/nodefense_system_message_banking.csv | tail -8
save('Week 11: real run B (no defense, system_message, banking)')

In [ ]:
%cd /kaggle/working/pi-detector-thesis
# C. Detector vs. the HELD-OUT attack style: the hardest, fairest test
!python scripts/run_baseline.py --detector {DETECTOR} --attack system_message --suites banking --out results/defended_{DETECTOR}_system_message_banking.csv | tail -10
save('Week 11: real run C (detector, system_message, banking)')

### Optional: the other three suites
Completes the comparison with your full week 6 baseline (29.3% attack success). About 900 more agent runs, a few hours.

In [ ]:
%cd /kaggle/working/pi-detector-thesis
for suite in ['slack', 'travel', 'workspace']:
    !python scripts/run_baseline.py --detector {DETECTOR} --suites {suite} --out results/defended_{DETECTOR}_important_instructions_{suite}.csv | tail -4
    save(f'Week 11: detector, important_instructions, {suite}')

## 8. The week 11 table

In [ ]:
%cd /kaggle/working/pi-detector-thesis
import glob
week6 = ','.join(f'results/nodefense_{s}.csv' for s in ['banking', 'slack', 'travel', 'workspace'])
runs = [f'No defense (week 6)={week6}']
def add(label, pattern):
    if glob.glob(pattern): runs.append(f'{label}={pattern}')
add('Detector', f'results/defended_{DETECTOR}_important_instructions_*.csv')
add('No defense', 'results/nodefense_system_message_*.csv')
add('Detector', f'results/defended_{DETECTOR}_system_message_*.csv')
for attack in ['important_instructions', 'system_message']:
    add('Gullible, no defense', f'results/week11_offline_{attack}_none.csv')
    add('Gullible + TF-IDF', f'results/week11_offline_{attack}_tfidf.csv')
    add('Gullible + detector', f'results/week11_offline_{attack}_{DETECTOR}.csv')
args = ' '.join(f'--run "{r}"' for r in runs)
!python scripts/summarize_end_to_end.py {args} > /dev/null
save('Week 11: end-to-end table')
Markdown(open('results/week11_end_to_end.md').read())

## 9. Before you close the session
Optional: zip the trained models and download them from the **Output** panel (needed only to rerun week 11 on your Mac). Then click the **power button** at the top to stop the session and save GPU hours.

In [ ]:
%cd /kaggle/working
!zip -qr models.zip pi-detector-thesis/models/deberta-v3-small pi-detector-thesis/models/deberta-v3-small-nochunk && echo 'ZIPPED: download models.zip from the Output panel'